In [0]:
%run "/Users/ajmalkhan88083@gmail.com/end_to_end_data_engineering/workflow/logging_config"

In [0]:
%run "/Users/ajmalkhan88083@gmail.com/end_to_end_data_engineering/schemas/customer_schema"

In [0]:
%run "/Users/ajmalkhan88083@gmail.com/end_to_end_data_engineering/schemas/order_schema"

In [0]:
%run "/Users/ajmalkhan88083@gmail.com/end_to_end_data_engineering/schemas/orders_items_schema"

In [0]:
%run "/Users/ajmalkhan88083@gmail.com/end_to_end_data_engineering/schemas/product_schema"

In [0]:
%run "/Users/ajmalkhan88083@gmail.com/end_to_end_data_engineering/datasource/reader_factory"


In [0]:
from pyspark.sql.functions import *
from pyspark.sql.functions import col, when, concat_ws
import logging

logger = logging.getLogger(__name__)

# Base class for data extraction
class Extractor:
    def __init__(self):
        pass

    def extract(self):
        # Abstract method to be implemented by subclasses
        pass

# ===================================================================
# EXTRACTOR - Extract Data from Source
# ===================================================================

# Concrete implementation to extract all required datasets
class ExtractorAllData(Extractor):

    # =====================================================
    # STEP 0 — DATA INGESTION
    # Purpose: Load raw CSV files into DataFrames
    # =====================================================

    def extract(self):
        try:
            logger.info("Starting data extraction...")

            # Load Customer dataset

            try:
                logger.info("  Reading customer.csv...")               
                customerInputDF = get_data_source(
                                data_type="csv",
                                file_path="/Volumes/workspace/data/raw_data/customer.csv",
                                schema=customer_schema
                ).get_data_frame()
                logger.info(f"Loaded {customerInputDF.count()} customer records")
            except Exception as e:
                logger.error(f" Failed to load customer.csv: {str(e)}")
                raise
            
            # Load Orders dataset
            try:                
                logger.info("  Reading orders.csv...")
                orderInputDF = get_data_source(
                               data_type="csv",
                               file_path="/Volumes/workspace/data/raw_data/orders.csv",
                               schema=order_schema
                ).get_data_frame()
                logger.info(f"Loaded {orderInputDF.count()} Order records")
            except Exception as e:
                logger.error(f" Failed to load order.csv: {str(e)}")
                raise

            # Load Order Items dataset
            try:
                logger.info("  Reading order_items.csv...")
                order_itemsInputDF = get_data_source(
                                    data_type="csv",
                                    file_path="/Volumes/workspace/data/raw_data/order_items.csv",
                                    schema=orders_items_schema
                ).get_data_frame()
                logger.info(f"Loaded {order_itemsInputDF.count()} order_items records")
            except Exception as e:
                logger.error(f" Failed to load order_items.csv: {str(e)}")
                raise

            # Load Product dataset
            try:
                logger.info("  Reading product.csv...")
                productInputDF = get_data_source(
                                data_type="csv",
                                file_path="/Volumes/workspace/data/raw_data/project.csv",
                                schema=product_schema
                ).get_data_frame()
                logger.info(f"Loaded {productInputDF.count()} product records")
            except Exception as e:
                logger.error(f" Failed to load product.csv: {str(e)}")
                raise

        except Exception as e:
            logger.error(f" Extraction failed: {str(e)}", exc_info=True)
            raise       

        # =====================================================
        # STEP 1 — INGESTION VALIDATION
        # Purpose:
        # Validate schema correctness and record counts
        # Helps confirm successful ingestion
        # =====================================================

        # customerInputDF.printSchema()
        # orderInputDF.printSchema()
        # order_itemsInputDF.printSchema()
        # productInputDF.printSchema()
        
        # print("Customer Row Count     :", customerInputDF.count())
        # print("Orders Row Count       :", orderInputDF.count())
        # print("Order Items Row Count  :", order_itemsInputDF.count())
        # print("Product Row Count      :", productInputDF.count())

        # =====================================================
        # STEP 2 — DATA PROFILING (NULL ANALYSIS)
        # Purpose:
        # Identify null distribution column-wise
        # Helps define data quality rules
        # =====================================================

        def null_profile(df, df_name):
            """
            Displays null count for each column in the DataFrame
            """
            print(f"Null Profile for {df_name}")
            df.select([
                count(when(col(c).isNull(), c)).alias(c)
                for c in df.columns
            ]).show(truncate=False)

        # null_profile(customerInputDF, "Customer")
        # null_profile(orderInputDF, "Order")
        # null_profile(order_itemsInputDF, "Order_items")
        # null_profile(productInputDF, "Product")

        # =====================================================
        # STEP 3 — DATA CLEANING (NULL HANDLING)
        # Purpose:
        # Replace null values with default business-friendly values
        # =====================================================

        # -------------------
        # Customer Dataset
        # -------------------
        # Fill null values with default placeholders
        customerInputDF = customerInputDF.fillna({
            'gender': 'Others',
            'city': 'Unknown',
            'age': -1,
            'first_name': 'Unknown',
            'last_name': 'Unknown'
        })

        # Create full_name if missing using first_name + last_name
        customerInputDF = customerInputDF.withColumn(
            'full_name',
            when(
                col('full_name').isNull(),
                concat_ws(' ', col('first_name'), col('last_name'))
            ).otherwise(col('full_name'))
        )

        # -------------------
        # Orders Dataset
        # -------------------
        # Fill null values with default placeholders
        orderInputDF = orderInputDF.fillna({
            'payment_method': 'Unknown',
            'order_status': 'Unknown',
            'city': 'Unknown',
            'state': 'Unknown'
        })

        # -------------------
        # Order Items Dataset
        # -------------------
        # Set default discount to 0.0 if null
        order_itemsInputDF = order_itemsInputDF.fillna({
            'discount': 0.0
        })

        # -------------------
        # Product Dataset
        # -------------------
        # Fill null values with default placeholders
        productInputDF = productInputDF.fillna({
            'brand': 'Unknown',
            'category': 'Unknown',
            'sub_category': 'Unknown'
        })

        # =====================================================
        # STEP 4 — DATA QUALITY FLAGGING
        # Purpose:
        # Apply business validation rules
        # Create is_valid_record flag
        # =====================================================

        # -------------------
        # Customer Validation Rules
        # -------------------
        # Mark record as valid if all critical fields meet business rules
        customerInputDF = customerInputDF.withColumn(
            'is_valid_record',
            when(
                col('customer_id').isNotNull()
                & col('signup_date').isNotNull()
                & (col('age') > 0)
                & (col('city') != 'Unknown'),
                True
            ).otherwise(False)
        )

        # -------------------
        # Orders Validation Rules
        # -------------------
        # Mark record as valid if all critical fields meet business rules
        orderInputDF = orderInputDF.withColumn(
            'is_valid_record',
            when(
                col('order_id').isNotNull()
                & col('customer_id').isNotNull()
                & col('order_date').isNotNull()
                & col('order_status').isNotNull()
                & (col('total_amount') >= 0),
                True
            ).otherwise(False)
        )

        # -------------------
        # Order Items Validation Rules
        # -------------------
        # Mark record as valid if all critical fields meet business rules
        order_itemsInputDF = order_itemsInputDF.withColumn(
            'is_valid_record',
            when(
                col('order_id').isNotNull()
                & col('product_id').isNotNull()
                & (col('quantity') > 0)
                & (col('unit_price') > 0)
                & (col('net_amount') > 0),
                True
            ).otherwise(False)
        )

        # -------------------
        # Product Validation Rules
        # -------------------
        # Mark record as valid if all critical fields meet business rules
        productInputDF = productInputDF.withColumn(
            'is_valid_record',
            when(
                col('product_id').isNotNull()
                & col('category').isNotNull()
                & (col('MRP') > 0),
                True
            ).otherwise(False)
        )

        # =====================================================
        # STEP 5 — SPLIT VALID & REJECTED RECORDS
        # Purpose:
        # Separate clean data from bad data
        # Enables downstream processing and auditing
        # =====================================================

        def split_valid_rejected(df, df_name):
            """
            Splits DataFrame into valid and rejected records
            based on is_valid_record flag and prints counts
            """
            valid_df = df.filter(col("is_valid_record") == True)
            rejected_df = df.filter(col("is_valid_record") == False)

            print(f"{df_name} DataFrame")
            print(f"Valid {df_name} Records    :", valid_df.count())
            print(f"Rejected {df_name} Records :", rejected_df.count())

            return valid_df, rejected_df

        # =====================================================
        # STEP 6 — SILVER LAYER WRITE
        # Purpose:
        # Persist only valid records into Silver Delta tables
        # Silver layer contains cleaned, validated data
        # =====================================================

        def silver_layer(df, df_name, database="silver"):
            """
            Filters valid records and writes them to the Silver layer
            as managed Delta tables.
            """
            # Only write valid records to Silver layer
            valid_df = df.filter(col("is_valid_record") == True)

            valid_df.write \
                .format("delta") \
                .mode("overwrite") \
                .saveAsTable(f"{database}.{df_name}_silver")

        # Write all datasets to Silver layer
        silver_layer(customerInputDF, "customer")
        silver_layer(orderInputDF, "order")
        silver_layer(order_itemsInputDF, "order_items")
        silver_layer(productInputDF, "product")

        # =====================================================
        # STEP 7 — DELTA TABLE INGESTION (SILVER)
        # Purpose:
        # Read curated Silver-layer Delta tables
        # for downstream transformation logic
        # =====================================================

        # Load Customer Silver Delta Table
        customer_silverInputDeltaDF = get_data_source(
            data_type="delta",
            file_path="silver.customer_silver",
        ).get_data_frame()

        # Load Orders Silver Delta Table
        order_silverInputDeltaDF = get_data_source(
            data_type = 'delta',
            file_path = 'silver.order_silver'
        ).get_data_frame()

        # Load Order Items Silver Delta Table
        order_items_silverInputDeltaDF = get_data_source(
            data_type = 'delta',
            file_path = 'silver.order_items_silver'
        ).get_data_frame()

        # Load Product Silver Delta Table
        product_items_silverInputDeltaDF = get_data_source(
            data_type = 'delta',
            file_path = 'silver.product_silver'
        ).get_data_frame()


        # Package required DataFrames for transformation layer
        inputDFs = {
            "order_silverInputDeltaDF": order_silverInputDeltaDF,
            "customer_silverInputDeltaDF": customer_silverInputDeltaDF,
            "order_items_silverInputDeltaDF":order_items_silverInputDeltaDF,
            "product_items_silverInputDeltaDF":product_items_silverInputDeltaDF
        }

        # Return extracted and curated DataFrames
        return inputDFs


# -------------------------------------------------------------------
# Entry point for standalone execution
# Purpose:
# Allows this extractor to be run independently for testing
# -------------------------------------------------------------------
if __name__ == "__main__":
    job = ExtractorAllData()
    job.extract()